In [2]:
import pandas as pd
print(pd.__version__)

3.0.2


In [3]:
df = pd.read_csv("../data/raw/boxing/boxing_matches.csv")

In [4]:
df.shape

(387427, 26)

In [5]:
df = df.drop(columns=["judge1_A", "judge1_B", "judge2_A", "judge2_B", "judge3_A", "judge3_B", "decision"])
df = df[df["result"] != "draw"]          
df.shape

(362655, 19)

In [6]:
df=df.drop(columns=["reach_A", "reach_B"])

In [7]:
df.shape

(362655, 17)

In [8]:
df.isna().sum()

age_A        31301
age_B       121933
height_A    127926
height_B    237481
stance_A    142387
stance_B    142387
weight_A    233250
weight_B    238410
won_A            0
won_B            0
lost_A           0
lost_B           0
drawn_A          0
drawn_B          0
kos_A            0
kos_B           77
result           0
dtype: int64

In [9]:
# Numériques — imputer par la médiane
for col in ["age_A", "age_B", "height_A", "height_B", "weight_A", "weight_B", "kos_B"]:
    df[col] = df[col].fillna(df[col].median())

# Catégorielles — imputer par le mode
for col in ["stance_A", "stance_B"]:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
df.isna().sum()

age_A       0
age_B       0
height_A    0
height_B    0
stance_A    0
stance_B    0
weight_A    0
weight_B    0
won_A       0
won_B       0
lost_A      0
lost_B      0
drawn_A     0
drawn_B     0
kos_A       0
kos_B       0
result      0
dtype: int64

In [11]:
df["age_diff"] = df["age_A"] - df["age_B"]
for base in ["age", "height", "weight", "won", "lost", "drawn", "kos"]:
    df[f"{base}_diff"] = df[f"{base}_A"] - df[f"{base}_B"]
    df["stance_diff"] = (df["stance_A"] != df["stance_B"]).astype(int)


In [12]:
df.head()

,age_A,age_B,height_A,height_B,stance_A,stance_B,weight_A,weight_B,won_A,won_B,...,kos_B,result,age_diff,stance_diff,height_diff,weight_diff,won_diff,lost_diff,drawn_diff,kos_diff
1,26.0,31.0,175.0,185.0,orthodox,orthodox,164.0,164.0,48,50,...,32.0,win_A,-5.0,0,-10.0,0.0,-2,-1,0,2.0
2,28.0,26.0,176.0,175.0,orthodox,orthodox,154.0,154.0,23,47,...,33.0,win_B,2.0,0,1.0,0.0,-24,-1,0,-20.0
3,25.0,29.0,175.0,174.0,orthodox,orthodox,155.0,155.0,46,31,...,19.0,win_A,-4.0,0,1.0,0.0,15,-2,1,13.0
4,25.0,35.0,175.0,170.0,orthodox,orthodox,155.0,140.0,45,40,...,33.0,win_A,-10.0,0,5.0,15.0,5,-3,1,-1.0
5,24.0,31.0,175.0,175.0,orthodox,orthodox,140.0,140.0,44,32,...,28.0,win_A,-7.0,0,0.0,0.0,12,0,1,3.0


In [14]:
df["label"] = (df["result"] == "win_B").astype(int)

In [15]:
df.shape

(362655, 26)

In [16]:
# Les colonnes finales qu'on garde pour le modèle
feature_cols = ["age_diff", "height_diff", "weight_diff", "won_diff", "lost_diff", "drawn_diff", "kos_diff", "stance_diff"]
df_clean = df[feature_cols + ["label"]]
df_clean.shape

(362655, 9)

In [17]:
df_clean.to_csv("../data/processed/boxing/boxing_cleaned.csv", index=False)

In [18]:
df.shape

(362655, 26)

In [19]:
df_clean["label"].value_counts(normalize=True)

label
0    0.886961
1    0.113039
Name: proportion, dtype: float64

In [20]:
df_mirror = df_clean.copy()

diff_cols = ["age_diff", "height_diff", "weight_diff", "won_diff", "lost_diff", "drawn_diff", "kos_diff"]
df_mirror[diff_cols] = -df_mirror[diff_cols]

df_mirror["label"] = 1 - df_mirror["label"]

df_full = pd.concat([df_clean, df_mirror], ignore_index=True)

In [21]:
df_full.shape
df_full["label"].value_counts(normalize=True)

label
0    0.5
1    0.5
Name: proportion, dtype: float64